# cli

> Single `kosha` entry point with subcommands for shell-based harnesses.
> Default output is readable markdown; pass `--as-json` for JSON (piping/harnesses).

In [ ]:
#| default_exp cli

In [ ]:
#| hide
from nbdev.showdoc import *

/Users/71293/code/personal/orgs/kosha/.venv/lib/python3.13/site-packages/fastprogress/fastprogress.py:171: UserWarning: Couldn't import ipython display functions, progress bar will use console behavior
  warn("Couldn't import ipython display functions, progress bar will use console behavior")


In [ ]:
#| export
import json, sys
from fastcore.script import call_parse, Param
from kosha.core import Kosha, env_pkg_versions

In [ ]:
#| export
def _to_json(obj):
    'Recursively convert kosha result objects to JSON-serializable types.'
    if isinstance(obj, (set, frozenset)): return sorted(obj)
    if isinstance(obj, dict): return {k: _to_json(v) for k, v in obj.items()}
    if hasattr(obj, '__iter__') and not isinstance(obj, str): return [_to_json(x) for x in obj]
    return obj

def _print_results(results):
    'Print context/search results as readable markdown blocks.'
    for r in results:
        m = r.get('metadata', {}) if isinstance(r, dict) else {}
        mod = m.get('mod_name', r.get('path', 'unknown')) if m else r.get('path', 'unknown')
        line = m.get('lineno', '?') if m else '?'
        pr = r.get('pagerank', 0) or 0
        print(f"\n## {mod}  (line {line})  [pagerank: {pr:.5f}]")
        content = (r.get('content', '') or '')[:300]
        if content: print(f"```\n{content}\n```")
        callers = r.get('callers', [])
        if callers: print(f"Callers: {', '.join(list(callers)[:5])}")

In [ ]:
# Test _to_json handles all expected types
from fastcore.foundation import L
assert _to_json({'a': {1, 2}}) == {'a': [1, 2]}
assert _to_json(L([{'x': 1}])) == [{'x': 1}]
assert _to_json('plain') == 'plain'
print("helpers ok")

helpers ok


In [ ]:
#| export
@call_parse
def sync(
    pkgs:str=None,       # comma-separated package names; defaults to all pyproject.toml deps
    parallel:bool=False, # run repo, env, and graph sync in parallel
    embed:bool=True,     # embed code chunks (set False for fast metadata-only update)
):
    'Sync repo + env packages + call graph into .kosha/ databases.'
    k = Kosha()
    pkg_list = pkgs.split(',') if pkgs else None
    k.sync(pkgs=pkg_list, in_parallel=parallel, embed=embed)

In [ ]:
#| export
@call_parse
def context(
    query:str,           # search query (supports key:value filters e.g. package:fastcore)
    limit:int=10,        # max results
    repo:bool=True,      # include repo results
    env:bool=True,       # include env results
    graph:bool=True,     # include call graph enrichment
    as_json:bool=False,  # output JSON instead of markdown
):
    'Fan-out semantic search over repo and env, optionally graph-enriched.'
    k = Kosha()
    results = k.context(query, limit=limit, repo=repo, env=env, graph=graph)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
# Test that functions are callable
assert callable(sync)
assert callable(context)
print("sync + context defined ok")

sync + context defined ok


In [ ]:
#| export
@call_parse
def repo_context(
    query:str,          # search query
    limit:int=10,       # max results
    as_json:bool=False, # output JSON
):
    'Semantic + keyword search over indexed repo code only.'
    k = Kosha()
    results = k.repo_context(query, limit=limit)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
#| export
@call_parse
def env_context(
    query:str,          # search query (package names auto-detected as filters)
    limit:int=10,       # max results
    as_json:bool=False, # output JSON
):
    'Semantic search over indexed env packages only.'
    k = Kosha()
    results = k.env_context(query, limit=limit)
    if as_json: print(json.dumps(_to_json(results)))
    else: _print_results(results)

In [ ]:
#| export
@call_parse
def ni(
    mod_name:str,       # fully-qualified module node name e.g. fastcore.basics.merge
    as_json:bool=False, # output JSON
):
    'Node info: callers, callees, co_dispatched, pagerank for a single graph node.'
    k = Kosha()
    result = k.ni(mod_name)
    if as_json: print(json.dumps(_to_json(result)))
    else:
        print(f"\n## {mod_name}")
        print(f"pagerank: {result.get('pagerank', 0):.5f}")
        callers = result.get('callers', [])
        callees = result.get('callees', [])
        co = result.get('co_dispatched', [])
        if callers: print(f"Callers ({len(callers)}): {', '.join(list(callers)[:10])}")
        if callees: print(f"Callees ({len(callees)}): {', '.join(list(callees)[:10])}")
        if co: print(f"Co-dispatched: {', '.join(list(co)[:10])}")

In [ ]:
#| export
@call_parse
def watch(
    dir:str=None,  # directory to watch; defaults to repo root
):
    "Live incremental re-index on file changes. Blocking \u2014 Ctrl-C to stop."
    from pathlib import Path as _Path
    k = Kosha()
    k.watch_repo(dir=_Path(dir) if dir else None)

In [ ]:
#| export
@call_parse
def public_api(
    pkg:str,            # package name e.g. "fastcore" or submodule "fastcore.basics"
    module:str=None,    # restrict to a submodule (overrides pkg for scoping)
    as_json:bool=False, # output JSON
):
    "List public API entries for a package (respects __all__ + @patch methods)."
    k = Kosha()
    target = module or pkg
    results = k.public_api(target)
    if as_json: print(json.dumps(_to_json(results)))
    else:
        print(f"\n## Public API: {target}  ({len(results)} entries)")
        for r in results:
            name = r.get('mod_name', '')
            doc = r.get('docstring', '') or ''
            suffix = f'  # {doc.splitlines()[0][:60]}' if doc.strip() else ''
            print(f"  {name}{suffix}")

In [ ]:
#| export
@call_parse
def api_paths(
    from_pkg:str,       # source package (call origin)
    to_pkg:str,         # target package (call destination)
    k:int=15,           # top-k API nodes per package to consider
    as_json:bool=False, # output JSON
):
    "Shortest call-graph paths from from_pkg public API to to_pkg public API."
    ko = Kosha()
    paths = ko.api_call_paths(from_pkg, to_pkg, k=k)
    if as_json: print(json.dumps(_to_json(paths)))
    else:
        print(f"\n## Call paths: {from_pkg} \u2192 {to_pkg}  ({len(paths)} targets reached)")
        if not paths: print("  (no paths found)")
        _arr = ' → '
        for target, path in sorted(paths.items(), key=lambda x: len(x[1])):
            print(f"\n  \u2192 {target}  (hops: {len(path)})")
            print(f"    {_arr.join(path)}")

In [ ]:
#| export
@call_parse
def dep_stack(
    seeds:str=None,     # comma-separated seed package names; defaults to pyproject.toml deps
    depth:int=1,        # BFS depth
    as_json:bool=False, # output JSON
):
    "BFS dependency layers from seed packages, ordered by coupling strength."
    k = Kosha()
    seed_list = seeds.split(',') if seeds else list(env_pkg_versions().keys())
    layers = k.dep_stack(seeds=seed_list, depth=depth)
    if as_json: print(json.dumps(_to_json(layers)))
    else:
        for i, layer in enumerate(layers):
            label = "seeds" if i == 0 else f"depth {i}"
            pkgs = sorted(layer) if isinstance(layer, (set, frozenset)) else layer
            print(f"\n## Layer {i} ({label}): {', '.join(pkgs)}")

In [ ]:
#| export
@call_parse
def top_nodes(
    pkg:str,            # package name e.g. "fastcore"
    k:int=5,            # number of top nodes to return
    as_json:bool=False, # output JSON
):
    "Top-k public API nodes for a package ranked by PageRank in the call graph."
    ko = Kosha()
    nodes = ko.top_nodes(pkg, k=k)
    if as_json: print(json.dumps(_to_json(nodes)))
    else:
        print(f"\n## Top {k} nodes: {pkg}")
        for node in nodes: print(f"  {node}")

In [ ]:
#| export
@call_parse
def status(as_json:bool=False):
    "Show index freshness: file/pkg/node counts, stale files, and stale packages."
    s = Kosha().status()
    if as_json: print(json.dumps(s))
    else:
        stale_f = f" ({s['stale_files']} stale)" if s['stale_files'] else ""
        stale_p = f"  stale pkgs: {s['stale_pkgs']}" if s['stale_pkgs'] else ""
        print(f"files: {s['files']}{stale_f}  packages: {s['packages']}  graph nodes: {s['graph_nodes']}{stale_p}")


In [ ]:
#| export
@call_parse
def where_to_add(
    description:str,    # what you want to add
    limit:int=5,        # max results
    as_json:bool=False, # output JSON
):
    "Find likely insertion points for new code matching description."
    results = Kosha().where_to_add(description, limit=limit)
    if as_json: print(json.dumps(_to_json(results)))
    else:
        for r in results:
            co = ', '.join(r['co_dispatched'][:3])
            print(f"  {r['path']}:{r['insert_after']}  ({r['node']})")
            if co: print(f"    peers: {co}")


In [ ]:
#| export
_DISPATCH = {
    'context':        lambda k, a: k.context(**a),
    'repo_context':   lambda k, a: k.repo_context(**a),
    'env_context':    lambda k, a: k.env_context(**a),
    'ni':             lambda k, a: k.ni(a['mod_name']),
    'top_nodes':      lambda k, a: k.top_nodes(**a),
    'public_api':     lambda k, a: k.public_api(**a),
    'api_call_paths': lambda k, a: k.api_call_paths(**a),
    'sync':           lambda k, a: (k.sync(**a), 'synced')[1],
    'status':         lambda k, a: k.status(),
    'where_to_add':   lambda k, a: k.where_to_add(**a),
}

@call_parse
def daemon():
    "Persistent kosha kernel. Reads newline-delimited JSON from stdin, writes results to stdout."
    k = Kosha()
    print(json.dumps({"ok": True, "status": "ready"}), flush=True)
    for line in sys.stdin:
        if not (line := line.strip()): continue
        try:
            req = json.loads(line)
            fn = _DISPATCH[req['cmd']]
            print(json.dumps({"ok": True, "result": _to_json(fn(k, req.get('args', {})))}), flush=True)
        except Exception as e:
            print(json.dumps({"ok": False, "error": str(e)}), flush=True)


In [ ]:
#| export
@call_parse
def install():
    "Install kosha SKILL.md to .agents/skills/kosha/ and .claude/skills/kosha/."
    from .core import mv_skill_md, repo_root
    mv_skill_md(dry_run=False, dir=repo_root())


In [ ]:
#| export
CMDS = {
    'sync':           sync,
    'context':        context,
    'repo-context':   repo_context,
    'env-context':    env_context,
    'ni':             ni,
    'watch':          watch,
    'public-api':     public_api,
    'api-paths':      api_paths,
    'dep-stack':      dep_stack,
    'top-nodes':      top_nodes,
    'daemon':         daemon,
    'status':         status,
    'where-to-add':   where_to_add,
    'install':         install,
}

def main():
    "Entry point for the `kosha` CLI command."
    if len(sys.argv) < 2 or sys.argv[1] not in CMDS:
        cmds = ' | '.join(CMDS)
        print(f"Usage: kosha [{cmds}]")
        sys.exit(0 if len(sys.argv) < 2 else 1)
    cmd = sys.argv.pop(1)
    CMDS[cmd]()


In [ ]:
# Test dispatcher has all expected commands
assert set(CMDS.keys()) == {
    'sync', 'context', 'repo-context', 'env-context', 'ni',
    'watch', 'public-api', 'api-paths', 'dep-stack', 'top-nodes',
    'daemon', 'status', 'where-to-add', 'install'
}
print(f"CMDS registered: {sorted(CMDS)}")


CMDS registered: ['api-paths', 'context', 'daemon', 'dep-stack', 'env-context', 'install', 'ni', 'public-api', 'repo-context', 'status', 'sync', 'top-nodes', 'watch', 'where-to-add']
